In [ ]:
%matplotlib inline

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from cnmf import cNMF
from IPython.display import Image
from matplotlib import pyplot as plt
from scipy.io import mmread
import anndata
rc_parms = {"figure.figsize": [5, 5], "figure.dpi": 500, "font.size": 10, "font.family": "Arial"}
save_parms = {"bbox_inches": "tight", "transparent": True}
if not os.path.exists("breast_cancer-Z1"):
    os.mkdir("breast_cancer-Z1")

np.random.seed(14)

In [ ]:
adata_gex = sc.read("../multiHIVE/outputs/breast_cancer_cite_HierarVI.h5ad")
adata_gex

In [ ]:
adata = sc.AnnData(adata_gex.obsm["RNA_Z1_denoised"])
adata.obsm["X_umap"] = adata_gex.obsm["Zc_umap"]
adata.obs = adata_gex.obs[["celltype_minor", "celltype_major", "batch"]]

In [ ]:
# adata.write("breast_cancer-Z1/counts.h5ad")

In [ ]:
numiter = 100  # Number of NMF replicates. Set this to a larger value ~200 for real data. We set this to a relatively low value here for illustration at a faster speed
numhvgenes = 2000  ## Number of over-dispersed genes to use for running the actual factorizations

## Results will be saved to [output_directory]/[run_name] which in this example is example_PBMC/cNMF/pbmc_cNMF
output_directory = "breast_cancer-Z1"
if not os.path.exists(output_directory):
    os.mkdir(output_directory)
run_name = "bc_cNMF"

K = " ".join([str(i) for i in range(5, 11)])

seed = 14  ## Specify a seed pseudorandom number generation for reproducibility

## Path to the filtered counts dataset we output previously
countfn = "breast_cancer-Z1/counts.h5ad"

In [ ]:
cnmf_obj = cNMF(output_dir=output_directory, name=run_name)

In [ ]:
cnmf_obj.prepare(counts_fn=countfn, components=np.arange(5, 11), n_iter=numiter, seed=14, num_highvar_genes=numhvgenes)

In [ ]:
cnmf_obj.factorize_multi_process(20)

In [ ]:
cnmf_obj.combine()

In [ ]:
cnmf_obj.k_selection_plot(close_fig=False)
print("This saves the corresponding figure to the following file: %s" % cnmf_obj.paths["k_selection_plot"])

In [ ]:
print('done')

In [ ]:
selected_K = 8
density_threshold = 0.02

In [ ]:
cnmf_obj.consensus(k=selected_K, density_threshold=density_threshold, show_clustering=True, close_clustergram_fig=False)

In [ ]:
! ls ./breast_cancer-Z1/bc_cNMF

In [ ]:
adata = sc.read(countfn)
adata.obsm["X_umap"] = adata_gex.obsm["Zc_umap"]
adata.obs = adata_gex.obs[["celltype_minor", "celltype_major", "batch"]]

In [ ]:
hvgs = open("./breast_cancer-Z1/bc_cNMF/bc_cNMF.overdispersed_genes.txt").read().split("\n")
hvgs

In [ ]:
adata.raw = sc.pp.log1p(adata.copy(), copy=True)

In [ ]:
adata

In [ ]:
adata = adata[:, hvgs]

In [ ]:
# sc.pp.scale(adata)
# sc.pp.pca(adata)
# sc.pl.pca_variance_ratio(adata, log=True)
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)

In [ ]:
usage_norm, gep_scores, gep_tpm, topgenes = cnmf_obj.load_results(K=selected_K, density_threshold=density_threshold)
usage_norm.columns = ["Program-%d" % i for i in usage_norm.columns]
# usage_file = cnmf_obj.paths['consensus_usages__txt'] % (selected_K, '0_8')
# gene_scores_file = cnmf_obj.paths['gene_spectra_score__txt'] % (selected_K, '0_8')
# gene_tpm_file = cnmf_obj.paths['gene_spectra_tpm__txt'] % (selected_K, '0_8')

In [ ]:
usage_norm

In [ ]:
# topgenes.to_csv("./breast_cancer-Z1/top_genes.csv")

In [ ]:
topgenes

In [ ]:
gep_scores

In [ ]:
adata.obs = pd.merge(left=adata.obs, right=usage_norm, how="left", left_index=True, right_index=True)

In [ ]:
to_plot = usage_norm.columns.to_list()
to_plot.extend(["celltype_minor", "celltype_major"])

In [ ]:
for val in to_plot:
    with plt.rc_context(rc_parms):
        sc.pl.umap(adata, color=val, vmin=0, vmax=1, show=False, frameon=False)
        plt.savefig("./breast_cancer-Z1/figures/" + val + ".png", **save_parms)

In [ ]:
adata.obs["celltype_major"].value_counts()

In [ ]:
adata_usage = anndata.AnnData(adata.obs[usage_norm.columns], obsm = {'X_umap':adata.obsm['X_umap']}, 
                              obs= adata.obs[['celltype_minor', 'celltype_major', 'batch']]
                             )
adata_usage.obs['celltype_major'] = adata_usage.obs['celltype_major'].cat.reorder_categories(['T-cells', 'B-cells', 'Myeloid', "PVL", 'Endothelial', 'Cancer Epithelial','CAFs', 'Plasmablasts'])
with plt.rc_context(rc_parms):
    sc.pl.heatmap(adata_usage, var_names = usage_norm.columns, groupby='celltype_major', show=False)
    plt.gca().set_ylabel('')
    plt.savefig("./breast_cancer-Z1/figures/Usages_heatmap.png", **save_parms)
    
    # sc.pl.heatmap(adata_usage, var_names = usage_norm.columns, groupby='celltype_major', dendrogram=True, show = False)
    # plt.savefig("./breast_cancer-Z1/figures/Usages_heatmap_dg.png", **save_parms)

In [ ]:
# save figures
# import re

# import matplotlib
# import matplotlib.pyplot as plt
# import scanpy as sc

# rc_parms = {"figure.figsize": [5, 5], "figure.dpi": 500, "font.size": 10, "font.family": "Arial"}
# save_parms = {"bbox_inches": "tight", "transparent": True}

# adata = sc.read_hcells5ad("./outputs/breast_cancer_cite_HierarVI.h5ad")
# for basis in ["Z1_umap", "Z1p_umap", "Z1r_umap", "Z2_umap", "Zc_umap"]:
#     for color in ["celltype_minor", "celltype_major", "batch"]:
#         with plt.rc_context(rc_parms):
#             sc.pl.embedding(adata, basis, color=color, show=False, frameon=False)
#             plt.savefig("./outputs/Embeddings/" + basis + "_" + color + ".png", **save_parms)